In [24]:
import glob
import json
import re
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import optuna

In [8]:
data_folder = 'data/regionwise_yearwise_samples/'

In [12]:
# 1. Discover all region-year files dynamically
# Pattern matches files like: {REGION}_{YEAR}_v{VERSION}.csv
file_paths = glob.glob(data_folder + "*_*_v*.csv")

region_files = {}
for p in file_paths:
    filename = os.path.basename(p)
    match = re.search(r"([A-Za-z0-9]+)_(\d{4})_v\d+", filename)
    if match:
        region, year = match.groups()
        region_files.setdefault(region, []).append((int(year), p))

print(f"Discovered regions: {list(region_files.keys())}")

Discovered regions: ['DES']


In [ ]:
# 2. Combine files per region, clean metadata, and save
combined_region_dfs = {}
output_dir = data_folder + "aggregated_regionwise_data"
os.makedirs(output_dir, exist_ok=True)

meta_cols = [
    "system:index", ".geo", "year", "region", "class",
    "lat", "lon", "area_m2", "area_proportion", "mosaic_version", "usable_count"
    "q1_count","q2_count", "q3_count", "q4_count", "quarters_present",
]

# Standard remapping rules applied to the dataset (Level 2 -> Level 1)
old_codes = [5, 3, 4, 12, 11, 66, 19, 36, 9, 41, 24, 23, 75, 30, 25, 33, 76, 34, 61]
new_codes = [5, 3, 4, 12, 11, 66, 19, 14, 14, 14, 22, 22, 22, 22, 22, 33, 76, 34, 61]
class_remap_dict = dict(zip(old_codes, new_codes))

for region, items in region_files.items():
    print(f"\nProcessing region: {region} ({len(items)} files)...")
    dfs_to_combine = []
    
    # Sort files chronologically by year
    for year, filepath in sorted(items, key=lambda x: x[0]):
        temp_df = pd.read_csv(filepath)
        
        # Ensure 'region' and 'year' tags exist
        temp_df["region"] = region
        if "year" not in temp_df.columns or temp_df["year"].isna().all():
            temp_df["year"] = year
            
        dfs_to_combine.append(temp_df)
    
    # Combine multi-year data for this region
    region_df = pd.concat(dfs_to_combine, ignore_index=True)

    # --- LEVEL 2 -> LEVEL 1 CLASS REMAPPING ---
    print(f"  Classes before remapping: {sorted(region_df['class'].unique())}")
    region_df["class"] = region_df["class"].replace(class_remap_dict).astype(int)
    print(f"  Classes after remapping:  {sorted(region_df['class'].unique())}")
    
    # Identify predictor feature columns
    feature_cols = [c for c in region_df.columns if c not in meta_cols]
    
    # Impute missing values (e.g., cloudy quarterly composite bands) per region
    imputer = SimpleImputer(strategy="median")
    region_df[feature_cols] = imputer.fit_transform(region_df[feature_cols])
    
    # Save combined regional dataset
    csv_out = os.path.join(output_dir, f"{region}_combined_multiyear.csv")
    parquet_out = os.path.join(output_dir, f"{region}_combined_multiyear.parquet")
    
    # region_df.to_csv(csv_out, index=False)
    region_df.to_parquet(parquet_out, index=False) #much smaller in size
    
    combined_region_dfs[region] = region_df
    print(f"Saved {region} dataset: {len(region_df):,} rows, {len(feature_cols)} features -> {parquet_out}")


Processing region: DES (4 files)...


/tmp/ipykernel_123819/3281160363.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  temp_df["region"] = region


Saved DES dataset: 469,507 rows, 113 features -> data/regionwise_yearwise_samples/aggregated_regionwise_data/DES_combined_multiyear.parquet


In [17]:
input_dir = output_dir
parquet_files = glob.glob(os.path.join(input_dir, "*_combined_multiyear.parquet"))

label_col = "class"

regional_study_packages = {}

def prune_correlated_features(df_train, feature_cols, importances_dict, corr_threshold=0.85):
    """
    Identifies feature pairs with Pearson |r| > corr_threshold.
    Between any correlated pair, drops the one with lower baseline feature importance.
    """
    # Sample 15,000 rows if dataset is large to compute correlation matrix rapidly
    sample_df = df_train[feature_cols]
    if len(sample_df) > 15000:
        sample_df = sample_df.sample(15000, random_state=42)
        
    corr_matrix = sample_df.corr().abs()
    
    # Upper triangle of correlation matrix
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    dropped_cols = set()
    for col1 in upper.columns:
        if col1 in dropped_cols:
            continue
        # Find features correlated with col1 exceeding threshold
        high_corr = upper.index[upper[col1] > corr_threshold].tolist()
        for col2 in high_corr:
            if col2 in dropped_cols:
                continue
            # Drop the feature with lower baseline importance
            if importances_dict[col1] >= importances_dict[col2]:
                dropped_cols.add(col2)
            else:
                dropped_cols.add(col1)
                break
                
    uncorrelated = [c for c in feature_cols if c not in dropped_cols]
    return uncorrelated

for p in parquet_files:
    region_name = os.path.basename(p).split("_combined_multiyear")[0]
    print(f"\n================ Processing Region: {region_name} ================")
    
    df_region = pd.read_parquet(p)
    feature_cols = [c for c in df_region.columns if c not in meta_cols]
    years_present = sorted(df_region["year"].unique())
    print(f"Total Rows: {len(df_region):,} | Years: {years_present} | Raw Features: {len(feature_cols)}")

    # 1. 80:20 Split PER YEAR (Locked validation folds)
    train_by_year = {}
    val_by_year = {}
    for yr in years_present:
        yr_data = df_region[df_region["year"] == yr].reset_index(drop=True)
        tr, val = train_test_split(
            yr_data, test_size=0.20, stratify=yr_data[label_col], random_state=42
        )
        train_by_year[yr] = tr.reset_index(drop=True)
        val_by_year[yr] = val.reset_index(drop=True)

    # 2. Baseline Model & Global Feature Importance
    pooled_train = pd.concat(train_by_year.values(), ignore_index=True)
    baseline_rf = RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42)
    baseline_rf.fit(pooled_train[feature_cols], pooled_train[label_col])
    
    # Store importances mapping
    feat_importances = dict(zip(feature_cols, baseline_rf.feature_importances_))
    all_ranked = sorted(feature_cols, key=lambda c: feat_importances[c], reverse=True)

    # 3. Collinearity Pruning (|r| > 0.85)
    uncorr_candidates = prune_correlated_features(
        pooled_train, feature_cols, feat_importances, corr_threshold=0.85
    )
    # Sort the uncorrelated features by baseline importance
    uncorr_ranked = sorted(uncorr_candidates, key=lambda c: feat_importances[c], reverse=True)
    
    print(f"  [All Features]: {len(all_ranked)} bands")
    print(f"  [Pruned Uncorrelated (|r| <= 0.85)]: {len(uncorr_ranked)} bands (dropped {len(all_ranked) - len(uncorr_ranked)} collinear features)")

    regional_study_packages[region_name] = {
        "train_by_year": train_by_year,
        "val_by_year": val_by_year,
        "feature_sets": {
            "all": all_ranked,
            "uncorrelated": uncorr_ranked
        }
    }


================ Processing Region: DES ================
Total Rows: 469,507 | Years: [np.int64(1995), np.int64(2005), np.int64(2015), np.int64(2024)] | Raw Features: 113
  [All Features]: 113 bands
  [Pruned Uncorrelated (|r| <= 0.85)]: 54 bands (dropped 59 collinear features)


In [20]:
# regional_study_packages

In [26]:
def make_regional_objective(region_name, pkg):
    train_by_year = pkg["train_by_year"]
    val_by_year = pkg["val_by_year"]
    feature_sets = pkg["feature_sets"]

    def objective(trial):
        # A. Feature Pool Selection
        feat_strategy = trial.suggest_categorical("feature_strategy", ["all", "uncorrelated"])
        
        if feat_strategy == "uncorrelated":
            # Use the entire set of pruned uncorrelated bands (no Top-K tuning)
            active_features = feature_sets["uncorrelated"]
            max_input_features = len(active_features)
        else:
            # For "all", tune how many top-ranked features to include
            ranked_all = feature_sets["all"]
            max_input_features = trial.suggest_int("max_input_features", 10, len(ranked_all))
            active_features = ranked_all[:max_input_features]

        # B. Downsample fraction of the 80% training data
        train_fraction = trial.suggest_float("train_fraction", 0.20, 1.0, step=0.20)

        # C. Hyperparameters
        # GEE smileRandomForest direct parameter mapping
        rf_params = {
            "n_estimators": trial.suggest_int("numberOfTrees", 50, 150, step=25),
            "min_samples_leaf": trial.suggest_int("minLeafPopulation", 2, 12),
            # "max_features": "sqrt",  # maps directly to GEE sqrt(variables)
            "max_depth": None,       # Unconstrained depth, matching GEE Smile behavior
            "n_jobs": -1,
            "random_state": 42
        }

        yearly_macro_f1s = []
        total_samples_trained = 0

        # D. Train dedicated model per year for this region
        for yr, yr_train in train_by_year.items():
            yr_val = val_by_year[yr]

            # Stratified subsample
            n_sub = int(len(yr_train) * train_fraction)
            sub_indices = (
                yr_train.groupby(label_col, group_keys=False)
                .apply(lambda grp: grp.sample(
                    n=max(3, int(n_sub * len(grp) / len(yr_train))),
                    random_state=42
                ))
                .index
            )

            X_sub = yr_train.loc[sub_indices, active_features]
            y_sub = yr_train.loc[sub_indices, label_col]
            total_samples_trained += len(sub_indices)

            clf = RandomForestClassifier(**rf_params)
            clf.fit(X_sub, y_sub)

            # Evaluate on locked validation fold
            preds = clf.predict(yr_val[active_features])
            score = f1_score(yr_val[label_col], preds, average="macro")
            yearly_macro_f1s.append(score)

        mean_macro_f1 = float(np.mean(yearly_macro_f1s))
        return mean_macro_f1, total_samples_trained

    return objective

In [22]:
data_folder

'data/regionwise_yearwise_samples/'

In [27]:
regional_results = {}
output_config_dir = data_folder + "optuna_regional_gee_configs"
os.makedirs(output_config_dir, exist_ok=True)

for region_name, pkg in regional_study_packages.items():
    print(f"\n================ Running Study: Region {region_name} ================")
    
    study = optuna.create_study(
        study_name=f"rf_{region_name}",
        directions=["maximize", "minimize"]
    )
    study.optimize(make_regional_objective(region_name, pkg), n_trials=40, n_jobs=1)
    
    pareto_trials = study.best_trials
    best_trial = max(pareto_trials, key=lambda t: t.values[0])
    
    best_strategy = best_trial.params["feature_strategy"]
    if best_strategy == "uncorrelated":
        selected_bands = pkg["feature_sets"]["uncorrelated"]
        num_bands = len(selected_bands)
    else:
        num_bands = best_trial.params["max_input_features"]
        selected_bands = pkg["feature_sets"]["all"][:num_bands]

    region_summary = {
            "region": region_name,
            "feature_strategy": best_strategy,
            "macro_f1": best_trial.values[0],
            "total_samples": best_trial.values[1],
            "train_fraction": best_trial.params["train_fraction"],
            "num_input_features": num_bands,
            "numberOfTrees": best_trial.params["numberOfTrees"],
            "variablesPerSplit": int(np.sqrt(num_bands)),
            "minLeafPopulation": best_trial.params["minLeafPopulation"],
            "selected_bands": selected_bands
        }
    
    regional_results[region_name] = region_summary
    
    config_path = os.path.join(output_config_dir, f"{region_name}_best_params.json")
    with open(config_path, "w") as f:
        json.dump(region_summary, f, indent=2)
        
    print(f"\n[Finished {region_name}]")
    print(f"  Chosen Strategy: {best_strategy} ({num_bands} features)")
    print(f"  Macro-F1: {region_summary['macro_f1']:.4f} | Total Samples: {region_summary['total_samples']:,}")
    print(f"  Config saved to -> {config_path}")

[I 2026-09-24 16:28:07,029] A new study created in memory with name: rf_DES



================ Running Study: Region DES ================


[I 2026-09-24 16:28:43,420] Trial 0 finished with values: [0.9492570334653045, 375603.0] and parameters: {'feature_strategy': 'uncorrelated', 'train_fraction': 1.0, 'numberOfTrees': 50, 'minLeafPopulation': 12}.
[I 2026-09-24 16:29:22,273] Trial 1 finished with values: [0.9035718694392136, 150199.0] and parameters: {'feature_strategy': 'uncorrelated', 'train_fraction': 0.4, 'numberOfTrees': 150, 'minLeafPopulation': 12}.
[I 2026-09-24 16:29:35,173] Trial 2 finished with values: [0.8397352278552103, 75081.0] and parameters: {'feature_strategy': 'all', 'max_input_features': 37, 'train_fraction': 0.2, 'numberOfTrees': 100, 'minLeafPopulation': 11}.
[I 2026-09-24 16:30:05,232] Trial 3 finished with values: [0.9316279902762623, 225332.0] and parameters: {'feature_strategy': 'uncorrelated', 'train_fraction': 0.6000000000000001, 'numberOfTrees': 75, 'minLeafPopulation': 10}.
[I 2026-09-24 16:30:47,429] Trial 4 finished with values: [0.9032290658477821, 150199.0] and parameters: {'feature_stra


[Finished DES]
  Chosen Strategy: all (14 features)
  Macro-F1: 0.9737 | Total Samples: 375,603.0
  Config saved to -> data/regionwise_yearwise_samples/optuna_regional_gee_configs/DES_best_params.json
